# BusinessGPT Reward Model: rubert-base pairwise ranker

Trains a small Russian BERT (`DeepPavlov/rubert-base-cased`) as a pairwise reward model on labeled preference pairs. The notebook validates the dataset, splits by prompt group, evaluates ranking quality, and saves a local artifact before any optional Hugging Face publication.

**Pipeline position:**

1. SFT (`training.ipynb`) → v16 SFT
2. Multi-candidate labeling → grouped preference pairs
3. **This notebook** → versioned reward-model candidate
4. Best-of-N evaluation: generate candidates, score with the RM, and compare the selected answer against the base sampling policy

**Why rubert-base:**
- The RM must run beside the 9B GGUF on a CPU deployment with limited memory.
- The preference dataset is still small enough that a larger ranker would add cost and overfitting risk before the baseline is established.

**Release gate**: held-out pairwise accuracy, prompt-level top-1, and score/response-token-length correlation must all pass their frozen thresholds. A passing model is still only a candidate until best-of-N improves the end-to-end evaluation.

**Requirements**: Kaggle GPU T4 (one GPU is sufficient). Hugging Face upload is disabled by default.

## 0. Install Dependencies

In [ ]:
%%capture
%pip install --no-cache-dir --upgrade "transformers==4.56.2" "trl==0.23.1" "datasets==4.1.1" "accelerate==1.10.1" "huggingface_hub==0.35.0"

In [ ]:
import accelerate
import datasets
import huggingface_hub
import transformers
import trl

print("Dependency versions:")
for package in (transformers, trl, datasets, accelerate, huggingface_hub):
    print(f"  {package.__name__}={package.__version__}")

## 1. Config

In [ ]:
# Freeze these before training. Change RUN_ID and RM_REPO for every candidate.
RUN_ID       = "v16-grouped-r1"
RM_BASE      = "DeepPavlov/rubert-base-cased"
RM_REPO      = "vXofi/businessgpt-reward-rubert-v16-grouped-r1"
PAIRS_FILE   = "preference_pairs_v16_multi.jsonl"

MAX_LEN          = 512
HELD_OUT_PROMPTS = 25    # whole prompt groups, never seen in train/validation
VAL_FRAC         = 0.20  # fraction of remaining prompt groups
SEED             = 42

# Frozen release thresholds. MRR and margins remain diagnostic metrics.
MIN_PAIRWISE_ACC    = 0.75
MIN_TOP1_ACC        = 0.60
MAX_ABS_LENGTH_CORR = 0.30

SAVE_DIR      = f"rm_rubert_{RUN_ID}"
PUSH_TO_HF    = False
HF_PRIVATE    = True

if PUSH_TO_HF and RM_REPO == "vXofi/businessgpt-reward-rubert":
    raise ValueError("Use a versioned RM_REPO; the existing production repository cannot be overwritten.")

print(f"Run:      {RUN_ID}")
print(f"RM base:  {RM_BASE}")
print(f"Local:    {SAVE_DIR}")
print(f"HF repo:  {RM_REPO} (push={PUSH_TO_HF}, private={HF_PRIVATE})")
print(f"Pairs:    {PAIRS_FILE}")

## 2. Load preference pairs

In [ ]:
import glob as _glob
import hashlib
import json
import os
from collections import Counter, defaultdict

_candidates = [
    f"/kaggle/input/businessgpt-eval/{PAIRS_FILE}",
    f"/kaggle/input/businessgpt-eval/eval/{PAIRS_FILE}",
    f"eval/{PAIRS_FILE}",
    PAIRS_FILE,
]
_pairs_path = next((p for p in _candidates if os.path.isfile(p)), None)
if _pairs_path is None:
    _matches = _glob.glob(f"/kaggle/input/**/{PAIRS_FILE}", recursive=True)
    if _matches:
        _pairs_path = _matches[0]
if _pairs_path is None:
    raise FileNotFoundError(
        f"{PAIRS_FILE} not found. Attach the businessgpt-eval Kaggle dataset."
    )

with open(_pairs_path, encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f if line.strip()]
with open(_pairs_path, "rb") as f:
    pairs_sha256 = hashlib.sha256(f.read()).hexdigest()
if not pairs:
    raise ValueError(f"No preference pairs found in {_pairs_path}")

REQUIRED_FIELDS = {"prompt", "chosen", "rejected", "prompt_id", "category"}

def assistant_content(messages, *, row_number, side):
    if not isinstance(messages, list) or not messages:
        raise ValueError(f"row {row_number}: {side} must be a non-empty message list")
    message = messages[-1]
    if not isinstance(message, dict):
        raise ValueError(f"row {row_number}: last {side} message must be an object")
    if message.get("role") != "assistant":
        raise ValueError(f"row {row_number}: last {side} message must have role=assistant")
    content = message.get("content")
    if not isinstance(content, str) or not content.strip():
        raise ValueError(f"row {row_number}: {side} response is empty")
    return content.strip()

seen_pairs = set()
prompt_owners = {}
group_facts = defaultdict(lambda: {"prompts": set(), "chosen": set(), "category": set(), "rejected": set()})
for row_number, pair in enumerate(pairs, start=1):
    missing = REQUIRED_FIELDS - pair.keys()
    if missing:
        raise ValueError(f"row {row_number}: missing fields {sorted(missing)}")
    if not isinstance(pair["prompt"], list) or not pair["prompt"]:
        raise ValueError(f"row {row_number}: prompt must be a non-empty message list")
    for message in pair["prompt"]:
        if not isinstance(message, dict):
            raise ValueError(f"row {row_number}: every prompt message must be an object")
        if message.get("role") not in {"system", "user", "assistant"}:
            raise ValueError(f"row {row_number}: invalid prompt role {message.get('role')!r}")
        if not isinstance(message.get("content"), str):
            raise ValueError(f"row {row_number}: prompt content must be text")

    prompt_id = pair["prompt_id"]
    if not isinstance(prompt_id, str) or not prompt_id.strip():
        raise ValueError(f"row {row_number}: prompt_id must be non-empty text")
    if not isinstance(pair["category"], str) or not pair["category"].strip():
        raise ValueError(f"row {row_number}: category must be non-empty text")
    chosen = assistant_content(pair["chosen"], row_number=row_number, side="chosen")
    rejected = assistant_content(pair["rejected"], row_number=row_number, side="rejected")
    if chosen == rejected:
        raise ValueError(f"row {row_number}: chosen and rejected responses are identical")

    prompt_key = json.dumps(pair["prompt"], ensure_ascii=False, sort_keys=True)
    previous_owner = prompt_owners.setdefault(prompt_key, prompt_id)
    if previous_owner != prompt_id:
        raise ValueError(f"row {row_number}: exact prompt text is shared by {previous_owner} and {prompt_id}")
    pair_key = (prompt_id, prompt_key, chosen, rejected)
    if pair_key in seen_pairs:
        raise ValueError(f"row {row_number}: duplicate preference pair for {prompt_id}")
    seen_pairs.add(pair_key)

    facts = group_facts[prompt_id]
    facts["prompts"].add(prompt_key)
    facts["chosen"].add(chosen)
    facts["category"].add(pair["category"])
    if rejected in facts["rejected"]:
        raise ValueError(f"row {row_number}: duplicate rejected candidate for {prompt_id}")
    facts["rejected"].add(rejected)

for prompt_id, facts in group_facts.items():
    if len(facts["prompts"]) != 1:
        raise ValueError(f"{prompt_id}: prompt text differs within one prompt group")
    if len(facts["chosen"]) != 1:
        raise ValueError(f"{prompt_id}: multiple chosen responses make top-1 undefined")
    if len(facts["category"]) != 1:
        raise ValueError(f"{prompt_id}: category differs within one prompt group")

if len(group_facts) <= HELD_OUT_PROMPTS + 4:
    raise ValueError(
        f"Only {len(group_facts)} prompt groups; need more than HELD_OUT_PROMPTS={HELD_OUT_PROMPTS} plus train/val groups"
    )

pair_categories = Counter(pair["category"] for pair in pairs)
prompt_categories = Counter(next(iter(facts["category"])) for facts in group_facts.values())
print(f"Validated {len(pairs)} pairs / {len(group_facts)} prompt groups from {_pairs_path}")
print(f"  dataset sha256:      {pairs_sha256}")
print(f"  pairs by category:   {dict(sorted(pair_categories.items()))}")
print(f"  prompts by category: {dict(sorted(prompt_categories.items()))}")
print(f"  pairs per prompt:    {dict(sorted(Counter(len(f['rejected']) for f in group_facts.values()).items()))}")

## 3. Serialize each pair as full chat template

Same `<|im_start|>{role}\n{content}<|im_end|>` format that the SFT model receives — so the RM scores inputs in the same shape it'll see at inference time. The RM doesn't need to *parse* the special tokens (rubert will subword-split them); it just learns to correlate the surface form with chosen/rejected labels.

In [ ]:
def serialize(prompt_messages, response_text):
    parts = [
        f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in prompt_messages
    ]
    parts.append(f"<|im_start|>assistant\n{response_text}<|im_end|>")
    return "\n".join(parts)


records = []
for row_number, pair in enumerate(pairs, start=1):
    chosen_response = assistant_content(pair["chosen"], row_number=row_number, side="chosen")
    rejected_response = assistant_content(pair["rejected"], row_number=row_number, side="rejected")
    records.append({
        "chosen": serialize(pair["prompt"], chosen_response),
        "rejected": serialize(pair["prompt"], rejected_response),
        "prompt_id": pair["prompt_id"],
        "category": pair["category"],
    })

# Avoid printing private chat contents into notebook outputs.
sample = records[0]
print(
    f"Serialization sanity: prompt_id={sample['prompt_id']}, "
    f"chosen_chars={len(sample['chosen'])}, rejected_chars={len(sample['rejected'])}"
)

## 4. Train / val / held-out split

In [ ]:
import random as _random

# Split prompt groups, never individual pairs. A four-candidate judgment
# creates related rows that must remain in the same split.
by_prompt = defaultdict(list)
for record in records:
    by_prompt[record["prompt_id"]].append(record)

ids_by_category = defaultdict(list)
for prompt_id, group in by_prompt.items():
    categories = {row["category"] for row in group}
    if len(categories) != 1:
        raise ValueError(f"{prompt_id}: inconsistent categories after serialization")
    ids_by_category[next(iter(categories))].append(prompt_id)

rng = _random.Random(SEED)
for category in sorted(ids_by_category):
    ids_by_category[category].sort()
    rng.shuffle(ids_by_category[category])

def stratified_take(group_ids, target, *, reserve_per_category):
    """Take a deterministic category-aware subset while preserving later splits."""
    total = sum(len(ids) for ids in group_ids.values())
    capacities = {
        category: max(0, len(ids) - reserve_per_category)
        for category, ids in group_ids.items()
    }
    if target > sum(capacities.values()):
        raise ValueError(
            f"Cannot select {target} groups while reserving {reserve_per_category} per category"
        )

    # Give every category with capacity one group, then fill proportionally.
    counts = {category: int(capacity > 0) for category, capacity in capacities.items()}
    if sum(counts.values()) > target:
        raise ValueError(f"Target {target} is too small to represent every available category")
    while sum(counts.values()) < target:
        eligible = [category for category in group_ids if counts[category] < capacities[category]]
        category = max(
            eligible,
            key=lambda name: (target * len(group_ids[name]) / total - counts[name], len(group_ids[name]), name),
        )
        counts[category] += 1

    selected = []
    remaining = {}
    for category in sorted(group_ids):
        count = counts[category]
        selected.extend(group_ids[category][:count])
        remaining[category] = group_ids[category][count:]
    rng.shuffle(selected)
    return selected, remaining

held_ids, remaining_by_category = stratified_take(
    ids_by_category,
    HELD_OUT_PROMPTS,
    reserve_per_category=2,
)
remaining_prompt_count = sum(len(ids) for ids in remaining_by_category.values())
val_group_count = max(1, round(VAL_FRAC * remaining_prompt_count))
val_ids, train_by_category = stratified_take(
    remaining_by_category,
    val_group_count,
    reserve_per_category=1,
)
train_ids = [prompt_id for ids in train_by_category.values() for prompt_id in ids]
rng.shuffle(train_ids)

held_out = [row for prompt_id in held_ids for row in by_prompt[prompt_id]]
val_records = [row for prompt_id in val_ids for row in by_prompt[prompt_id]]
train_records = [row for prompt_id in train_ids for row in by_prompt[prompt_id]]

assert train_ids and val_ids and held_ids
assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(held_ids)
assert set(val_ids).isdisjoint(held_ids)
assert set(train_ids) | set(val_ids) | set(held_ids) == set(by_prompt)

def split_category_counts(prompt_ids):
    return dict(sorted(Counter(by_prompt[prompt_id][0]["category"] for prompt_id in prompt_ids).items()))

print(f"Train:    {len(train_records)} pairs / {len(train_ids)} prompts {split_category_counts(train_ids)}")
print(f"Val:      {len(val_records)} pairs / {len(val_ids)} prompts {split_category_counts(val_ids)}")
print(f"Held-out: {len(held_out)} pairs / {len(held_ids)} prompts {split_category_counts(held_ids)}")

## 5. Load model + tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(RM_BASE)
# Long chat contexts get truncated; we want to PRESERVE the recent response
# (which is what the RM judges) and drop the front of the conversation.
tokenizer.truncation_side = "left"
print(f"Tokenizer: {RM_BASE}, truncation_side=left, max_len={MAX_LEN}")

model = AutoModelForSequenceClassification.from_pretrained(
    RM_BASE,
    num_labels=1,
    trust_remote_code=True,
)
print(f"Model: {model.__class__.__name__}, params={sum(p.numel() for p in model.parameters()):,}")

### Truncation sanity check

Verify the truncation actually preserves the `<|im_start|>assistant\n...<|im_end|>` suffix — losing the response itself would defeat the reward signal.

In [ ]:
ASSISTANT_MARKER = "<|im_start|>assistant\n"

def contains_subsequence(sequence, subsequence):
    width = len(subsequence)
    return any(sequence[index:index + width] == subsequence for index in range(len(sequence) - width + 1))

def serialized_response(text):
    if ASSISTANT_MARKER not in text:
        raise ValueError("Serialized record has no assistant marker")
    return text.rsplit(ASSISTANT_MARKER, 1)[1].split("<|im_end|>", 1)[0]

marker_ids = tokenizer(ASSISTANT_MARKER, add_special_tokens=False)["input_ids"]
truncation_failures = []
truncated_count = 0
longest_tokens = 0
for record in records:
    for side in ("chosen", "rejected"):
        text = record[side]
        raw_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        kept_ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_LEN,
        )["input_ids"]
        response_ids = tokenizer(serialized_response(text), add_special_tokens=False)["input_ids"]
        response_tail = response_ids[-min(16, len(response_ids)):]
        longest_tokens = max(longest_tokens, len(raw_ids))
        truncated_count += int(len(raw_ids) > MAX_LEN)
        if not contains_subsequence(kept_ids, marker_ids) or not contains_subsequence(kept_ids, response_tail):
            truncation_failures.append((record["prompt_id"], side, len(raw_ids), len(response_ids)))

if truncation_failures:
    raise ValueError(
        "Truncation removed the assistant marker or response tail; "
        f"first failures: {truncation_failures[:5]}"
    )

print(f"Truncation audit passed for {len(records) * 2} serialized candidates")
print(f"  longest serialization: {longest_tokens} tokens")
print(f"  truncated candidates:  {truncated_count}")
print(f"  assistant marker and response tail preserved in every candidate")

## 6. Train

In [ ]:
from datasets import Dataset
from trl import RewardConfig, RewardTrainer
import torch

def trainer_rows(rows):
    return [{"chosen": row["chosen"], "rejected": row["rejected"]} for row in rows]

train_ds = Dataset.from_list(trainer_rows(train_records))
val_ds = Dataset.from_list(trainer_rows(val_records))

NUM_EPOCHS = 4
BATCH_SIZE = 8
LR         = 2e-5
STEPS_PER_EPOCH = max(1, len(train_ds) // BATCH_SIZE)

# RewardTrainer handles the pairwise loss internally: it scores chosen and rejected
# separately, applies -logsigmoid(score_chosen - score_rejected), backprops once.
rm_args = RewardConfig(
    output_dir=f"{SAVE_DIR}_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=max(1, int(STEPS_PER_EPOCH * NUM_EPOCHS * 0.1)),
    lr_scheduler_type="cosine",
    max_length=MAX_LEN,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

# TRL 0.23 expects this dict on HF model classes; some Transformers models lack it.
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

trainer = RewardTrainer(
    model=model,
    args=rm_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print(f"Training config:")
print(f"  Examples: {len(train_ds)} train / {len(val_ds)} val")
print(f"  Epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, lr={LR}, max_len={MAX_LEN}, fp16={torch.cuda.is_available()}")
trainer.train()
print(f"\nFinal train metrics:")
for k, v in trainer.state.log_history[-1].items():
    print(f"  {k}: {v}")

## 7. Held-out ranking gate

All frozen checks must pass:

- pairwise accuracy ≥ `MIN_PAIRWISE_ACC`
- prompt-level top-1 ≥ `MIN_TOP1_ACC`
- absolute score/response-token-length Pearson correlation ≤ `MAX_ABS_LENGTH_CORR`

MRR, score margins, confidence intervals, and per-category results are diagnostics. Passing this gate permits publication as a candidate; it does not prove that best-of-N improves end-to-end answer quality.

In [ ]:
device = next(model.parameters()).device
model.eval()

@torch.no_grad()
def score(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    return_tensors="pt").to(device)
    return model(**enc).logits.squeeze().item()


score_cache = {}
def cached_score(text):
    if text not in score_cache:
        score_cache[text] = score(text)
    return score_cache[text]

correct = 0
margins = []
held_groups = defaultdict(list)
category_pair_results = defaultdict(lambda: [0, 0])
for r in held_out:
    held_groups[r["prompt_id"]].append(r)
    s_chosen = cached_score(r["chosen"])
    s_rejected = cached_score(r["rejected"])
    if s_chosen > s_rejected:
        correct += 1
        category_pair_results[r["category"]][0] += 1
    category_pair_results[r["category"]][1] += 1
    margins.append(s_chosen - s_rejected)

top1_correct = 0
reciprocal_ranks = []
candidate_lengths = []
candidate_scores = []
category_top1_results = defaultdict(lambda: [0, 0])
def response_length(serialized):
    assistant = serialized.rsplit("<|im_start|>assistant\n", 1)[-1]
    response = assistant.split("<|im_end|>", 1)[0]
    return len(tokenizer(response, add_special_tokens=False)["input_ids"])

for prompt_id, group in held_groups.items():
    chosen_texts = {row["chosen"] for row in group}
    if len(chosen_texts) != 1:
        raise ValueError(f"{prompt_id}: held-out group has multiple chosen responses")
    chosen_text = next(iter(chosen_texts))
    rejected_texts = sorted({row["rejected"] for row in group})
    category = group[0]["category"]
    chosen_score = cached_score(chosen_text)
    rejected_scores = [cached_score(text) for text in rejected_texts]
    if chosen_score > max(rejected_scores, default=float("-inf")):
        top1_correct += 1
        category_top1_results[category][0] += 1
    category_top1_results[category][1] += 1
    rank = 1 + sum(value > chosen_score for value in rejected_scores)
    reciprocal_ranks.append(1 / rank)
    for text in [chosen_text, *rejected_texts]:
        candidate_lengths.append(response_length(text))
        candidate_scores.append(cached_score(text))

def pearson(xs, ys):
    mean_x = sum(xs) / len(xs)
    mean_y = sum(ys) / len(ys)
    numerator = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    denominator = (sum((x - mean_x) ** 2 for x in xs) * sum((y - mean_y) ** 2 for y in ys)) ** 0.5
    return numerator / denominator if denominator else 0.0

def wilson_interval(successes, total, z=1.96):
    if not total:
        return (0.0, 0.0)
    proportion = successes / total
    denominator = 1 + z * z / total
    center = (proportion + z * z / (2 * total)) / denominator
    radius = z * ((proportion * (1 - proportion) / total + z * z / (4 * total * total)) ** 0.5) / denominator
    return (center - radius, center + radius)

acc = correct / len(held_out)
top1_acc = top1_correct / len(held_groups)
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
length_score_corr = pearson(candidate_lengths, candidate_scores)
mean_margin = sum(margins) / len(margins)
median_margin = sorted(margins)[len(margins) // 2]
pairwise_ci = wilson_interval(correct, len(held_out))
top1_ci = wilson_interval(top1_correct, len(held_groups))

print(f"Held-out pairwise accuracy: {correct}/{len(held_out)} = {acc:.3f} (95% CI {pairwise_ci[0]:.3f}-{pairwise_ci[1]:.3f})")
print(f"Prompt-level top-1: {top1_correct}/{len(held_groups)} = {top1_acc:.3f} (95% CI {top1_ci[0]:.3f}-{top1_ci[1]:.3f})")
print(f"Mean reciprocal rank: {mrr:.3f}")
print(f"Score/response-token-length Pearson r: {length_score_corr:.3f}")
print(f"Margin (score_chosen - score_rejected): mean={mean_margin:.3f}, median={median_margin:.3f}")
print()
print("Per-category diagnostics (small categories are directional only):")
for category in sorted(category_pair_results):
    pair_correct, pair_total = category_pair_results[category]
    rank_correct, rank_total = category_top1_results[category]
    print(
        f"  {category}: pairwise={pair_correct}/{pair_total} ({pair_correct / pair_total:.3f}), "
        f"top1={rank_correct}/{rank_total} ({rank_correct / rank_total:.3f})"
    )

gate_checks = {
    "pairwise_accuracy": (acc >= MIN_PAIRWISE_ACC, acc, f">= {MIN_PAIRWISE_ACC:.2f}"),
    "prompt_top1": (top1_acc >= MIN_TOP1_ACC, top1_acc, f">= {MIN_TOP1_ACC:.2f}"),
    "abs_token_length_correlation": (
        abs(length_score_corr) <= MAX_ABS_LENGTH_CORR,
        abs(length_score_corr),
        f"<= {MAX_ABS_LENGTH_CORR:.2f}",
    ),
}
print("\nRelease gate:")
for name, (passed, value, threshold) in gate_checks.items():
    print(f"  {'PASS' if passed else 'FAIL'} {name}: {value:.3f} ({threshold})")
GATE_PASSED = all(result[0] for result in gate_checks.values())
print(f"\n{'GATE PASSED: candidate may be published for best-of-N evaluation.' if GATE_PASSED else 'GATE FAILED: keep the local artifact for analysis; do not publish as a candidate.'}")

## 8. Save locally and optionally publish

The trained artifact and metrics are always written to Kaggle output. Hugging Face publication requires both a passing gate and the explicit `PUSH_TO_HF=True` opt-in.

In [ ]:
import os

os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

metrics = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "base_model": RM_BASE,
    "pairs_file": PAIRS_FILE,
    "pairs_sha256": pairs_sha256,
    "seed": SEED,
    "max_length": MAX_LEN,
    "splits": {
        "train": {"pairs": len(train_records), "prompts": len(train_ids), "categories": split_category_counts(train_ids)},
        "validation": {"pairs": len(val_records), "prompts": len(val_ids), "categories": split_category_counts(val_ids)},
        "held_out": {"pairs": len(held_out), "prompts": len(held_ids), "categories": split_category_counts(held_ids)},
    },
    "held_out": {
        "pairwise_accuracy": acc,
        "pairwise_wilson_95": list(pairwise_ci),
        "prompt_top1": top1_acc,
        "prompt_top1_wilson_95": list(top1_ci),
        "mrr": mrr,
        "score_response_token_length_pearson": length_score_corr,
        "mean_margin": mean_margin,
        "median_margin": median_margin,
    },
    "gate": {
        "passed": GATE_PASSED,
        "thresholds": {
            "min_pairwise_accuracy": MIN_PAIRWISE_ACC,
            "min_prompt_top1": MIN_TOP1_ACC,
            "max_abs_token_length_correlation": MAX_ABS_LENGTH_CORR,
        },
        "checks": {name: passed for name, (passed, _, _) in gate_checks.items()},
    },
}
with open(f"{SAVE_DIR}/eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

readme = f"""---
library_name: transformers
tags: [reward-model, russian, rubert]
base_model: {RM_BASE}
---

# BusinessGPT Reward Model Candidate

Run `{RUN_ID}`, trained on {len(train_records)} grouped preference pairs (dataset SHA-256 `{pairs_sha256}`).

## Held-out evaluation

- Pairwise accuracy ({len(held_out)} pairs): **{acc:.3f}** (95% CI {pairwise_ci[0]:.3f}-{pairwise_ci[1]:.3f})
- Prompt-level top-1 ({len(held_groups)} prompts): **{top1_acc:.3f}** (95% CI {top1_ci[0]:.3f}-{top1_ci[1]:.3f})
- Mean reciprocal rank: **{mrr:.3f}**
- Score/response-token-length Pearson r: **{length_score_corr:.3f}**
- Margin (chosen - rejected): mean={mean_margin:.3f}, median={median_margin:.3f}
- Offline release gate: **{'passed' if GATE_PASSED else 'failed'}**

Passing the offline gate only qualifies this model for end-to-end best-of-N evaluation.

## Use

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tok = AutoTokenizer.from_pretrained("{RM_REPO}")
tok.truncation_side = "left"
mdl = AutoModelForSequenceClassification.from_pretrained("{RM_REPO}")

parts = [f"<|im_start|>{{m['role']}}\\n{{m['content']}}<|im_end|>" for m in prompt_messages]
parts.append(f"<|im_start|>assistant\\n{{response}}<|im_end|>")
text = "\\n".join(parts)
enc = tok(text, truncation=True, max_length={MAX_LEN}, return_tensors="pt")
score = mdl(**enc).logits.squeeze().item()
```
"""
with open(f"{SAVE_DIR}/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print(f"Saved model and metrics to /kaggle/working/{SAVE_DIR}")

if PUSH_TO_HF and not GATE_PASSED:
    print("PUSH_TO_HF=True, but the release gate failed: publication skipped and local artifact retained.")
elif PUSH_TO_HF:
    from huggingface_hub import HfApi, login
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=RM_REPO, private=HF_PRIVATE, exist_ok=False)
    api.upload_folder(
        folder_path=SAVE_DIR,
        repo_id=RM_REPO,
        commit_message=f"Reward model candidate {RUN_ID}",
    )
    print(f"Pushed new repository: https://huggingface.co/{RM_REPO}")
else:
    print("PUSH_TO_HF=False: local artifact retained; no network publication attempted.")